# Week 9 — Block 2: Guided Demo (Maps I)

**DATS 6401 · Visualization of Complex Data**

~30 min, fully offline on the **synthetic county data in `data/`** (20 polygons + a stats CSV — same pipeline as real data, zero download friction):

1. Load geometry; the CRS check (~6 min)
2. Join attributes; counts vs **rates** (~10 min)
3. Classification schemes, disclosed (~8 min)
4. The spatial join (~6 min)

*(For real-world geometry, swap in any GeoJSON — US counties, world countries; pipeline is identical.)*

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

gdf = gpd.read_file("data/counties.geojson")
print(len(gdf), "polygons | CRS:", gdf.crs)
gdf.head(3)

**The CRS ritual, narrated:** check `gdf.crs` FIRST, always. Our file declares EPSG:4326. With real data you'd now `to_crs(epsg=5070)` (equal-area) before any area math — say it even though our toy grid doesn't need it.

In [ ]:
# ALWAYS make the CRS explicit — silent assumptions are the #1 geo bug
gdf = gdf.set_crs(epsg=4326, allow_override=True)

stats = pd.read_csv("data/county_stats.csv")
joined = gdf.merge(stats, on="county_id")     # attribute join: plain merge, on the key
joined[["county_id", "name", "population", "events"]].head(3)

## Part 2 — Counts lie; rates compare

In [ ]:
joined["rate"] = joined["events"] / joined["population"] * 1000

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
joined.plot(column="events", cmap="OrRd", legend=True, ax=axes[0],
            edgecolor="white", linewidth=0.5)
axes[0].set_title("COUNTS: the big counties 'win'"); axes[0].axis("off")
joined.plot(column="rate", cmap="OrRd", legend=True, ax=axes[1],
            edgecolor="white", linewidth=0.5)
axes[1].set_title("RATE per 1,000: three true hotspots emerge"); axes[1].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# The receipts: top county by COUNT vs by RATE — different counties!
print("top by events:", joined.nlargest(3, "events")["name"].tolist())
print("top by rate:  ", joined.nlargest(3, "rate")["name"].tolist())

**Narrate:** the count map is a population map in disguise — and the printout proves it. (This synthetic data has three genuine hotspots; only the rate map finds them.)

## Part 3 — Classification, disclosed

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, scheme in zip(axes, ["quantiles", "equal_interval"]):
    joined.plot(column="rate", scheme=scheme, k=5, cmap="OrRd", legend=True,
                ax=ax, edgecolor="white", linewidth=0.5,
                legend_kwds={"fontsize": 7})
    ax.set_title(f'scheme="{scheme}", k=5'); ax.axis("off")
plt.suptitle("Same rates, two legends, two 'maps' — the scheme goes IN the title/legend")
plt.tight_layout(); plt.show()

## Part 4 — The spatial join: merge-by-location

In [ ]:
import numpy as np
from shapely.geometry import Point

rng = np.random.default_rng(7)
incidents = gpd.GeoDataFrame(
    geometry=[Point(rng.uniform(0, 5), rng.uniform(0, 4)) for _ in range(60)],
    crs="EPSG:4326")

with_county = gpd.sjoin(incidents, joined[["county_id", "name", "geometry"]],
                        predicate="within")
per_county = with_county.groupby("name").size().sort_values()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
joined.plot(ax=axes[0], color="#EAF2F6", edgecolor="#2E6E8E", linewidth=0.5)
incidents.plot(ax=axes[0], color="#d9534f", markersize=10)
axes[0].set_title("60 incident points over the counties"); axes[0].axis("off")
per_county.tail(8).plot.barh(ax=axes[1], color="#2E6E8E")
axes[1].set_title("sjoin → groupby: incidents per county")
plt.tight_layout(); plt.show()

**Narrate:** `sjoin(points, polygons, predicate="within")` is `merge` where the key is *geometry* — and it silently requires both layers in the same CRS (it always comes back to CRS).

## Wrap-up → Block 3

Pipeline: **read → CRS → join → rate → disclosed scheme → sjoin.** Yours next — with the choices written down.